In [27]:
from ultralytics import YOLO
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms
from pathlib import Path
import numpy as np
import torch
from PIL import Image

In [17]:
MASK_DIR = Path.home() / "data" / "SatelliteSegmentation" / "masks"
IMAGE_DIR = Path.home() / "data" / "SatelliteSegmentation" / "images"

print("Imágenes encontradas:", len(list(IMAGE_DIR.glob("*.png"))))
print("Máscaras encontradas:", len(list(MASK_DIR.glob("*.png"))))

Imágenes encontradas: 790
Máscaras encontradas: 790


In [18]:
#Se tiene que corregir las etiquetas de las clases

class_names = {
    0: "Sin etiqueta",
    10: "Árboles",
    20: "Matorrales",
    30: "Pastizales",
    40: "Cultivos",
    50: "Zonas construidas",
    60: "Suelo desnudo / vegetación escasa",
    70: "Nieve y hielo",
    80: "Cuerpos de agua",
    90: "Humedales",
    95: "Manglares",
    100: "Musgos y líquenes"
}

label_mapping = {
    0: 255,
    10: 0,
    20: 1,
    30: 2,
    40: 3,
    50: 4,
    60: 5,
    70: 6,
    80: 7,
    90: 8,
    95: 9,
    100: 10
}

image_files = sorted(IMAGE_DIR.glob("*.png"))
mask_files = sorted(MASK_DIR.glob("*_mask.png"))

images_by_name = {f.stem: f for f in image_files}
masks_by_name = {
    f.stem.replace("_mask", ""): f
    for f in mask_files
}

common_names = sorted(set(images_by_name) & set(masks_by_name))

print("Imágenes:", len(image_files))
print("Máscaras:", len(mask_files))
print("Pares encontrados:", len(common_names))

Imágenes: 790
Máscaras: 790
Pares encontrados: 790


In [26]:
images = []
masks = []

for name in common_names:
    image = Image.open(images_by_name[name]).convert("RGB")
    image = np.array(image, dtype=np.float32) / 255.0
    mask = np.array(Image.open(masks_by_name[name]))

    mapped_mask = np.full(mask.shape, 255, dtype=np.uint8)

    for original, mapped in label_mapping.items():
        mapped_mask[mask == original] = mapped

    images.append(torch.from_numpy(image).permute(2,0,1))
    masks.append(torch.from_numpy(mapped_mask.astype(np.int64)))

X = torch.stack(images)
Y = torch.stack(masks)


print("Dimensiones de imágenes:", X.shape)
print("Dimensiones de máscaras:", Y.shape)
print("Tipo de imágenes:", X.dtype)
print("Tipo de máscaras:", Y.dtype)   

Dimensiones de imágenes: torch.Size([790, 3, 256, 256])
Dimensiones de máscaras: torch.Size([790, 256, 256])
Tipo de imágenes: torch.float32
Tipo de máscaras: torch.int64


In [34]:
dataset = TensorDataset(X,Y)

#Qué hueva descargar test_train_split

train_size = int(len(dataset) * 0.70)
val_size = int(len(dataset) * 0.15)

test_size = len(dataset) - (train_size + val_size)

print(train_size)
print(test_size)
print(val_size)

553
119
118
